# Abstract Base Classes (ABCs) in Python

This notebook is a **from-scratch to intermediate/advanced** guide on Python's **Abstract Base Classes (ABCs)** using the `abc` module.

You already know OOP, so we will focus specifically on **why, when, and how to use ABCs**.

We will use **trading-related examples** throughout:
- **Orders** (market, limit, stop)
- **Positions** and **risk checks**
- **Market data feeds** (tick/quotes, historical data)

The notebook is organized as follows:

1. **Motivation and core concepts**
2. **Defining and using simple ABCs**
3. **Abstract methods, properties, and classmethods**
4. **Concrete methods and the Template Method pattern**
5. **Registering virtual subclasses**
6. **Mixins and multiple inheritance with ABCs**
7. **A trading case study: Order and Position hierarchies**
8. **A more advanced case study: MarketDataFeed and risk engine**
9. **Design tips, pitfalls, and best practices**

> You are encouraged to **run and edit the code cells**, try variations, and add your own experiments.


### Contents

- [1. Motivation: Why Abstract Base Classes?](#1-motivation-why-abstract-base-classes)
- [2. Basics: Defining and Using ABCs](#2-basics-defining-and-using-abcs)
- [3. Abstract Methods, Properties, and Classmethods](#3-abstract-methods-properties-and-classmethods)
- [4. Concrete Methods and the Template Method Pattern](#4-concrete-methods-and-the-template-method-pattern)
- [5. Virtual Subclasses and `register`](#5-virtual-subclasses-and-register)
- [6. Mixins and Multiple Inheritance with ABCs](#6-mixins-and-multiple-inheritance-with-abcs)
- [7. Trading Case Study I: Orders and Positions](#7-trading-case-study-i-orders-and-positions)
- [8. Trading Case Study II: Market Data Feeds and a Risk Engine](#8-trading-case-study-ii-market-data-feeds-and-a-risk-engine)
- [9. Design Tips, Pitfalls, and Best Practices](#9-design-tips-pitfalls-and-best-practices)

## 1. Motivation: Why Abstract Base Classes?

You already know OOP concepts like **inheritance**, **polymorphism**, and **interfaces**. ABCs are Python's way to make those ideas **explicit and enforceable at runtime**.

### 1.1 The problem: implicit interfaces

Imagine a simple trading system where you have different kinds of **orders**:

- `MarketOrder`
- `LimitOrder`
- `StopOrder`

Each order must implement a method like `execute(broker)`.

Without ABCs, you might just define separate classes that *happen* to implement `execute`. Nothing enforces this structurally.

### 1.2 What ABCs give you

**Abstract Base Classes provide:**

- **Explicit contracts**: "Any subclass of `BaseOrder` must implement `execute`."
- **Early error detection**: Instantiating a class that doesn't implement required methods raises `TypeError`.
- **Shared logic**: You can put shared concrete methods in the ABC.
- **Documentation**: The ABC itself documents the expected interface.

In trading systems, where multiple components must agree on interfaces (e.g., order types, risk checks, data feeds), ABCs help keep the design **clean and robust** as the system grows.

In [ ]:
# 2.1 Basic ABC example: a minimal trading order interface

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Protocol, runtime_checkable


class BaseOrder(ABC):
    """Abstract base class for all trading orders.

    Any concrete order must implement `execute`.
    """

    @abstractmethod
    def execute(self, broker) -> None:
        """Send the order to the broker.

        Subclasses must implement this.
        """
        raise NotImplementedError


# This will fail: we cannot instantiate BaseOrder directly
try:
    o = BaseOrder()  # TypeError
except TypeError as e:
    print("Cannot instantiate BaseOrder directly:", e)


Cannot instantiate BaseOrder directly: Can't instantiate abstract class BaseOrder without an implementation for abstract method 'execute'


: 

## 2. Basics: Defining and Using ABCs

### 2.1 `ABC` and `@abstractmethod`

Key pieces from `abc`:

- **`ABC`**: Base class for defining abstract base classes.
- **`@abstractmethod`**: Marks methods that *must* be overridden in concrete subclasses.

Rules:

- A class **inherits from `ABC`** (directly or indirectly) and has at least one `@abstractmethod` → it's an **abstract class**.
- You **cannot instantiate** an abstract class.
- A subclass becomes **concrete** (instantiable) only when **all** abstract methods are implemented.

Let's implement concrete order types that subclass `BaseOrder`.

In [2]:
# 2.2 Concrete implementations of BaseOrder

from dataclasses import dataclass


@dataclass
class MarketOrder(BaseOrder):
    symbol: str
    quantity: float  # positive for buy, negative for sell

    def execute(self, broker) -> None:
        print(f"Executing MARKET order via {broker}: {self.quantity} {self.symbol}")


@dataclass
class LimitOrder(BaseOrder):
    symbol: str
    quantity: float
    limit_price: float

    def execute(self, broker) -> None:
        print(
            f"Executing LIMIT order via {broker}: {self.quantity} {self.symbol} @ {self.limit_price}"
        )


# Now these are instantiable and can be used polymorphically
orders: list[BaseOrder] = [
    MarketOrder("AAPL", 100),
    LimitOrder("MSFT", -50, 320.0),
]

for order in orders:
    order.execute(broker="DemoBroker")

Executing MARKET order via DemoBroker: 100 AAPL
Executing LIMIT order via DemoBroker: -50 MSFT @ 320.0


### 2.3 Why not just use duck typing?

In pure duck typing, you might just **assume** that anything passed as `order` has an `execute` method. This is flexible but:

- You might accidentally pass an object without `execute`, discovering the bug only at runtime.
- You don't have a single canonical place that documents the expected **order interface**.

ABCs give you:

- A **formal base class** (`BaseOrder`).
- **Runtime enforcement** at instantiation time if required methods are missing.
- A natural place to put **shared logic** and **documentation**.

Next, we extend the idea to **abstract properties** and **class-level information** for orders.

## 3. Abstract Methods, Properties, and Classmethods

ABCs are not limited to regular instance methods. You can also have:

- **Abstract properties** → subclasses must provide those properties.
- **Abstract classmethods** / staticmethods → subclasses must implement them too.

In trading, this is useful when different instruments or order types must expose
some **metadata** or **configuration** in a uniform way.

In [3]:
# 3.1 Adding abstract properties and classmethods to orders

from abc import ABC, abstractmethod, abstractproperty


class Instrument(ABC):
    """Abstract representation of a tradable instrument."""

    @property
    @abstractmethod
    def symbol(self) -> str:
        """Ticker symbol, e.g. 'AAPL' or 'ESZ4'."""
        raise NotImplementedError

    @property
    @abstractmethod
    def tick_size(self) -> float:
        """Minimum price increment for the instrument."""
        raise NotImplementedError

    @classmethod
    @abstractmethod
    def from_dict(cls, data: dict) -> "Instrument":
        """Create an instrument from raw metadata (e.g. from an API)."""
        raise NotImplementedError


class Equity(Instrument):
    def __init__(self, symbol: str, tick_size: float = 0.01):
        self._symbol = symbol
        self._tick_size = tick_size

    @property
    def symbol(self) -> str:
        return self._symbol

    @property
    def tick_size(self) -> float:
        return self._tick_size

    @classmethod
    def from_dict(cls, data: dict) -> "Equity":
        return cls(symbol=data["symbol"], tick_size=data.get("tick_size", 0.01))


class Future(Instrument):
    def __init__(self, symbol: str, tick_size: float, contract_month: str):
        self._symbol = symbol
        self._tick_size = tick_size
        self.contract_month = contract_month

    @property
    def symbol(self) -> str:
        return self._symbol

    @property
    def tick_size(self) -> float:
        return self._tick_size

    @classmethod
    def from_dict(cls, data: dict) -> "Future":
        return cls(
            symbol=data["symbol"],
            tick_size=data["tick_size"],
            contract_month=data["contract_month"],
        )


instruments: list[Instrument] = [
    Equity("AAPL"),
    Future("ES", tick_size=0.25, contract_month="Z4"),
]

for inst in instruments:
    print(inst.symbol, inst.tick_size, type(inst).__name__)


AAPL 0.01 Equity
ES 0.25 Future


### 3.2 Notes on abstract properties

- Use `@property` **above** `@abstractmethod` (or `@abstractproperty` in older code).
- Subclasses must provide a concrete `@property` implementation.
- This is especially useful when you want to expose **read-only attributes** that are
  conceptually required by the interface, without forcing a particular storage layout.

### 3.3 Abstract classmethods

`@classmethod` + `@abstractmethod` is a good combination when you want each subclass
to define **how it is constructed from external data** (e.g., API responses, config files).

This keeps the parsing/normalization logic close to the concrete type.

## 4. Concrete Methods and the Template Method Pattern

ABCs can also contain **fully implemented methods**. This lets you:

- Put **shared logic** in the base class.
- Leave only the **varying steps** as abstract methods.

This is often known as the **Template Method pattern**: the ABC defines a high-level
algorithm, and subclasses fill in specific steps.

For trading, imagine we want a standard workflow for **submitting an order**:

1. Validate the order.
2. Run risk checks.
3. Transform to broker-specific format.
4. Actually send it.

Different order types may implement some steps differently, but the **overall flow**
should be consistent.

### Abstract hooks

**All abstract hooks are abstract methods, but not all abstract methods are hooks.**
Abstract hook is:
 
- An abstract method (often with a single underscore prefix, e.g., `_build_payload`) to suggest intended use as a "protected" or internal method; not part of the public API.
- Meant to be *overridden by subclasses* to inject "steps" or "specializations" into a concrete algorithm defined in the base class (here: `execute()`).
- Called by the non-abstract method (the Template Method) to let subclasses customize specific parts of the workflow without changing the overall process.

So, just as you said: it’s an abstract method, usually "private" by convention, intended for use within instance methods of the class (such as within `execute()`).

In [4]:
# 4.1 Order ABC with template method

from abc import ABC, abstractmethod
from dataclasses import dataclass


class Order(ABC):
    """Abstract order with a standard submission workflow.

    Subclasses supply the details; `submit` fixes the overall flow.
    """

    @abstractmethod
    def validate(self) -> None:
        """Raise an exception if the order is invalid."""
        raise NotImplementedError

    @abstractmethod
    def to_broker_payload(self) -> dict:
        """Return a dict ready to send to the broker API."""
        raise NotImplementedError

    @property
    @abstractmethod
    def description(self) -> str:
        """Human-readable description of the order."""
        raise NotImplementedError

    def submit(self, broker) -> None:
        """Template method: the high-level submission algorithm.

        Subclasses control only the variable parts (validate/payload),
        not the overall sequence.
        """
        print(f"Submitting order: {self.description}")

        # 1. Validate order-specific constraints
        self.validate()

        # 2. Risk checks (could call into another component)
        print("  Running basic risk checks...")

        # 3. Transform to broker payload
        payload = self.to_broker_payload()
        print("  Payload:", payload)

        # 4. Send to broker (simplified for this example)
        broker.send(payload)
        print("  Done.\n")


@dataclass
class SimpleBroker:
    name: str

    def send(self, payload: dict) -> None:
        # In real life this would hit an API; here we just print
        print(f"[{self.name}] Sending to exchange: {payload}")


@dataclass
class MarketOrder2(Order):
    symbol: str
    quantity: float

    def validate(self) -> None:
        if self.quantity == 0:
            raise ValueError("Quantity must not be zero")

    def to_broker_payload(self) -> dict:
        # Map our domain object to a generic broker payload
        return {
            "type": "market",
            "symbol": self.symbol,
            "qty": self.quantity,
        }

    @property
    def description(self) -> str:
        side = "BUY" if self.quantity > 0 else "SELL"
        return f"{side} {abs(self.quantity)} {self.symbol} @ market"


@dataclass
class LimitOrder2(Order):
    symbol: str
    quantity: float
    limit_price: float

    def validate(self) -> None:
        if self.quantity == 0:
            raise ValueError("Quantity must not be zero")
        if self.limit_price <= 0:
            raise ValueError("Limit price must be positive")

    def to_broker_payload(self) -> dict:
        return {
            "type": "limit",
            "symbol": self.symbol,
            "qty": self.quantity,
            "price": self.limit_price,
        }

    @property
    def description(self) -> str:
        side = "BUY" if self.quantity > 0 else "SELL"
        return f"{side} {abs(self.quantity)} {self.symbol} @ {self.limit_price} limit"


broker = SimpleBroker("DemoBroker2")

orders: list[Order] = [
    MarketOrder2("AAPL", 100),
    LimitOrder2("MSFT", -50, 320.0),
]

for o in orders:
    # Polymorphic call: same API, different concrete behavior
    o.submit(broker)


Submitting order: BUY 100 AAPL @ market
  Running basic risk checks...
  Payload: {'type': 'market', 'symbol': 'AAPL', 'qty': 100}
[DemoBroker2] Sending to exchange: {'type': 'market', 'symbol': 'AAPL', 'qty': 100}
  Done.

Submitting order: SELL 50 MSFT @ 320.0 limit
  Running basic risk checks...
  Payload: {'type': 'limit', 'symbol': 'MSFT', 'qty': -50, 'price': 320.0}
[DemoBroker2] Sending to exchange: {'type': 'limit', 'symbol': 'MSFT', 'qty': -50, 'price': 320.0}
  Done.



### 4.2 Takeaways from the template method example

- **`submit` is concrete** and lives in the ABC. It encodes the *policy* and *sequence*.
- Subclasses only implement **`validate`**, **`to_broker_payload`**, and **`description`**.
- Callers only depend on the `Order` interface and call `order.submit(broker)`.

This pattern is powerful in trading systems where you want a **consistent lifecycle** for orders, positions, or risk checks, while still allowing specialization.

## 5. Virtual Subclasses and `register`

Sometimes you **cannot** (or do not want to) make a class explicitly inherit from your ABC, but you still want it to be treated as a subtype.

For example:

- You use a third-party library that defines its own `OrderLike` class.
- You want to treat that class as an implementation of your `Order` interface **without modifying** its source.

ABCs support this with **virtual subclasses** via `MyABC.register(SomeClass)`.

Key points:

- The class **does not** need to inherit from the ABC.
- `isinstance(obj, MyABC)` and `issubclass(SomeClass, MyABC)` will return `True`.
- **No enforcement** happens here: it's your responsibility to ensure the class actually implements the required interface (duck typing).

In [5]:
# 5.1 Virtual subclass example

from abc import ABC, abstractmethod


class ExecutionReportSink(ABC):
    """Receives execution reports from the broker.

    For example, after an order is filled, partially filled, or cancelled.
    """

    @abstractmethod
    def on_execution_report(self, report: dict) -> None:
        raise NotImplementedError


class PrintSink:
    """A simple sink that just prints execution reports."""

    def on_execution_report(self, report: dict) -> None:
        print("[PRINT SINK]", report)


# Register PrintSink as a virtual subclass
ExecutionReportSink.register(PrintSink)


sink = PrintSink()
print("isinstance(sink, ExecutionReportSink)?", isinstance(sink, ExecutionReportSink))

# This is purely duck-typed: we trust that PrintSink implements on_execution_report
sink.on_execution_report({"order_id": 1, "status": "FILLED"})


isinstance(sink, ExecutionReportSink)? True
[PRINT SINK] {'order_id': 1, 'status': 'FILLED'}


### 5.2 When to use virtual subclasses

Use `register` when:

- You **cannot change** the class to inherit from the ABC (e.g., it's from a library).
- You want to integrate **existing classes** into your ABC-based type hierarchy.

Be aware:

- `register` does **not** enforce the presence of abstract methods at instantiation.
- This is effectively **duck typing with better introspection** (e.g., `isinstance`).

If you want both **enforcement** and **clarity**, prefer **explicit inheritance**.
If you just need tagging and interoperability, `register` can be very useful.

## 6. Mixins and Multiple Inheritance with ABCs

ABCs work well with **multiple inheritance** and **mixins**.

- A **mixin** is a class that provides *reusable behavior* (methods, properties),
  but is usually not meant to stand alone.
- When combined with ABCs, mixins can express **orthogonal capabilities**:
  - e.g. "this order can be cancelled", "this object can be risk-checked".

Python's method resolution order (MRO) handles multiple inheritance, but you should
keep mixins **small and focused**.

In [6]:
# 6.1 A cancellable mixin for orders

from abc import ABC, abstractmethod
from dataclasses import dataclass, field


class Cancellable(ABC):
    """Mixin/ABC for things that can be cancelled.

    Only defines the capability; does not dictate how cancellation works.
    """

    @abstractmethod
    def cancel(self, broker) -> None:
        raise NotImplementedError


@dataclass
class BaseOrderWithId(ABC):
    # Broker-assigned ID, filled in after sending the order
    order_id: int | None = field(default=None, init=False)

    @property
    @abstractmethod
    def description(self) -> str:  # similar to previous example
        raise NotImplementedError


@dataclass
class MarketOrder3(BaseOrderWithId, Cancellable):
    symbol: str
    quantity: float

    def execute(self, broker) -> None:
        payload = {"type": "market", "symbol": self.symbol, "qty": self.quantity}
        # Save the ID so we can later cancel
        self.order_id = broker.send_and_return_id(payload)
        print(f"Executed {self.description}, order_id={self.order_id}")

    def cancel(self, broker) -> None:
        if self.order_id is None:
            raise RuntimeError("Cannot cancel: order not sent yet")
        broker.cancel(self.order_id)
        print(f"Cancelled {self.description}, order_id={self.order_id}")

    @property
    def description(self) -> str:
        side = "BUY" if self.quantity > 0 else "SELL"
        return f"{side} {abs(self.quantity)} {self.symbol} @ market"


@dataclass
class LimitOrder3(BaseOrderWithId, Cancellable):
    symbol: str
    quantity: float
    limit_price: float

    def execute(self, broker) -> None:
        payload = {
            "type": "limit",
            "symbol": self.symbol,
            "qty": self.quantity,
            "price": self.limit_price,
        }
        self.order_id = broker.send_and_return_id(payload)
        print(f"Executed {self.description}, order_id={self.order_id}")

    def cancel(self, broker) -> None:
        if self.order_id is None:
            raise RuntimeError("Cannot cancel: order not sent yet")
        broker.cancel(self.order_id)
        print(f"Cancelled {self.description}, order_id={self.order_id}")

    @property
    def description(self) -> str:
        side = "BUY" if self.quantity > 0 else "SELL"
        return f"{side} {abs(self.quantity)} {self.symbol} @ {self.limit_price} limit"


@dataclass
class AdvancedBroker:
    name: str
    _next_id: int = field(default=1, init=False)

    def send_and_return_id(self, payload: dict) -> int:
        # Simulate sending the order and returning an ID
        order_id = self._next_id
        self._next_id += 1
        print(f"[{self.name}] Sending: {payload} (assigned id={order_id})")
        return order_id

    def cancel(self, order_id: int) -> None:
        print(f"[{self.name}] Cancelling order {order_id}")


broker = AdvancedBroker("AdvancedBroker1")

# We can type this as a list of Cancellable because both orders support cancel()
orders: list[Cancellable] = [
    MarketOrder3("AAPL", 100),
    LimitOrder3("MSFT", -50, 320.0),
]

for o in orders:
    # type: ignore[attr-defined]  # mypy-style hint: we know execute exists
    o.execute(broker)  # each order is also a Cancellable

for o in orders:
    o.cancel(broker)


[AdvancedBroker1] Sending: {'type': 'market', 'symbol': 'AAPL', 'qty': 100} (assigned id=1)
Executed BUY 100 AAPL @ market, order_id=1
[AdvancedBroker1] Sending: {'type': 'limit', 'symbol': 'MSFT', 'qty': -50, 'price': 320.0} (assigned id=2)
Executed SELL 50 MSFT @ 320.0 limit, order_id=2
[AdvancedBroker1] Cancelling order 1
Cancelled BUY 100 AAPL @ market, order_id=1
[AdvancedBroker1] Cancelling order 2
Cancelled SELL 50 MSFT @ 320.0 limit, order_id=2


### 6.2 Notes on mixins with ABCs

- Mixins like `Cancellable` express **capabilities** separate from the core type.
- Multiple inheritance lets you *compose* behavior: `BaseOrderWithId` + `Cancellable`.
- Keep mixins:
  - **Small** (few methods, focused purpose).
  - **Stateless or simple-state** where possible.

In larger trading systems, you might have mixins like:

- `RiskCheckable`
- `Persistable`
- `Serializable`

all combined with core ABCs for orders, positions, or feeds.

## 7. Trading Case Study I: Orders and Positions

Now we build a slightly more realistic **mini order/position system** that uses ABCs.

We want:

- An `AbstractOrder` that supports:
  - Validation
  - Conversion to a broker payload
  - Knowing its **instrument** and **side** (long/short)
- Concrete order types: `MarketOrder`, `LimitOrder`, `StopOrder`.
- A `Position` ABC that can be **updated** by fills and used to compute **PnL**.

This is not production-ready, but shows how ABCs structure the design.

In [7]:
# 7.1 Order and position ABCs for a mini trading system

from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Literal

Side = Literal["BUY", "SELL"]


@dataclass
class OrderInstrument(ABC):
    symbol: str

    @property
    @abstractmethod
    def tick_size(self) -> float:
        # Different instrument types (equity, future, option) provide their own tick size
        raise NotImplementedError


@dataclass
class EquityInstrument(OrderInstrument):
    tick: float = 0.01

    @property
    def tick_size(self) -> float:
        return self.tick


class AbstractOrder(ABC):
    """Base interface for all orders in this mini system."""

    instrument: OrderInstrument
    side: Side
    quantity: float

    @abstractmethod
    def validate(self) -> None:
        raise NotImplementedError

    @abstractmethod
    def to_broker_payload(self) -> dict:
        raise NotImplementedError

    @property
    @abstractmethod
    def description(self) -> str:
        raise NotImplementedError


@dataclass
class MarketOrderCS(AbstractOrder):  # CS = case study
    instrument: OrderInstrument
    side: Side
    quantity: float

    def validate(self) -> None:
        if self.quantity <= 0:
            raise ValueError("Quantity must be positive")

    def to_broker_payload(self) -> dict:
        # Minimal generic representation that a broker adapter could consume
        return {
            "type": "market",
            "symbol": self.instrument.symbol,
            "side": self.side,
            "qty": self.quantity,
        }

    @property
    def description(self) -> str:
        return f"{self.side} {self.quantity} {self.instrument.symbol} @ market"


@dataclass
class LimitOrderCS(AbstractOrder):
    instrument: OrderInstrument
    side: Side
    quantity: float
    limit_price: float

    def validate(self) -> None:
        if self.quantity <= 0:
            raise ValueError("Quantity must be positive")
        if self.limit_price <= 0:
            raise ValueError("Limit price must be positive")

    def to_broker_payload(self) -> dict:
        return {
            "type": "limit",
            "symbol": self.instrument.symbol,
            "side": self.side,
            "qty": self.quantity,
            "price": self.limit_price,
        }

    @property
    def description(self) -> str:
        return (
            f"{self.side} {self.quantity} {self.instrument.symbol} @ {self.limit_price} limit"
        )


@dataclass
class StopOrderCS(AbstractOrder):
    instrument: OrderInstrument
    side: Side
    quantity: float
    stop_price: float

    def validate(self) -> None:
        if self.quantity <= 0:
            raise ValueError("Quantity must be positive")
        if self.stop_price <= 0:
            raise ValueError("Stop price must be positive")

    def to_broker_payload(self) -> dict:
        return {
            "type": "stop",
            "symbol": self.instrument.symbol,
            "side": self.side,
            "qty": self.quantity,
            "stop": self.stop_price,
        }

    @property
    def description(self) -> str:
        return (
            f"{self.side} {self.quantity} {self.instrument.symbol} stop @ {self.stop_price}"
        )


# Position ABC


class Position(ABC):
    instrument: OrderInstrument

    @property
    @abstractmethod
    def quantity(self) -> float:
        raise NotImplementedError

    @property
    @abstractmethod
    def avg_price(self) -> float:
        """Average entry price."""
        raise NotImplementedError

    @abstractmethod
    def apply_fill(self, side: Side, qty: float, price: float) -> None:
        """Update the position state given a fill."""
        raise NotImplementedError

    @abstractmethod
    def unrealized_pnl(self, mark_price: float) -> float:
        raise NotImplementedError


@dataclass
class SimplePosition(Position):
    instrument: OrderInstrument
    _quantity: float = 0.0
    _avg_price: float = 0.0

    @property
    def quantity(self) -> float:
        return self._quantity

    @property
    def avg_price(self) -> float:
        return self._avg_price

    def apply_fill(self, side: Side, qty: float, price: float) -> None:
        if qty <= 0:
            raise ValueError("Fill quantity must be positive")

        # BUY fills increase quantity; SELL fills decrease it
        signed_qty = qty if side == "BUY" else -qty
        new_qty = self._quantity + signed_qty

        if self._quantity == 0 or (self._quantity > 0) == (signed_qty > 0):
            # Increasing in same direction → update average price
            total_cost = self._avg_price * abs(self._quantity) + price * abs(signed_qty)
            total_qty = abs(self._quantity) + abs(signed_qty)
            self._avg_price = total_cost / total_qty
        # else: reducing or flipping position → leave avg_price as is (simplified)

        self._quantity = new_qty

    def unrealized_pnl(self, mark_price: float) -> float:
        # PnL is mark-to-market vs average entry price
        return (mark_price - self._avg_price) * self._quantity


# Demo

inst = EquityInstrument("AAPL")

orders: list[AbstractOrder] = [
    MarketOrderCS(inst, side="BUY", quantity=100),
    LimitOrderCS(inst, side="SELL", quantity=50, limit_price=190.0),
]

for o in orders:
    o.validate()
    print(o.description, "->", o.to_broker_payload())

pos = SimplePosition(inst)

# Simulate some fills (opening then partially closing the position)
pos.apply_fill("BUY", qty=100, price=180.0)
pos.apply_fill("SELL", qty=50, price=185.0)

print("Position quantity:", pos.quantity)
print("Average price:", pos.avg_price)
print("Unrealized PnL @ 190:", pos.unrealized_pnl(190.0))


BUY 100 AAPL @ market -> {'type': 'market', 'symbol': 'AAPL', 'side': 'BUY', 'qty': 100}
SELL 50 AAPL @ 190.0 limit -> {'type': 'limit', 'symbol': 'AAPL', 'side': 'SELL', 'qty': 50, 'price': 190.0}
Position quantity: 50.0
Average price: 180.0
Unrealized PnL @ 190: 500.0


### 7.2 What ABCs buy you in this case study

- A clear **`AbstractOrder` contract**: validate, broker payload, description.
- A clear **`Position` contract**: how fills are applied, how PnL is computed.
- Separation between **instrument details** (`OrderInstrument`) and **orders/positions**.

As your system grows (e.g., adding options, futures, FX), you can:

- Add new concrete `OrderInstrument` subclasses.
- Add new `AbstractOrder` implementations (e.g., `IcebergOrder`, `BracketOrder`).
- Keep **callers** depending only on the **abstract interfaces**.

## 8. Trading Case Study II: Market Data Feeds and a Risk Engine

Now we look at **market data** and a small **risk engine**.

Goals:

- Define a `MarketDataFeed` ABC for streaming prices.
- Have multiple concrete implementations:
  - `SimulatedFeed` (driven by an in-memory price series)
  - `LiveFeed` (placeholder, e.g. wrapping an exchange API)
- Define a `RiskEngine` ABC that consumes orders and market data to enforce limits.

This shows how ABCs coordinate **different subsystems**: data, orders, and risk.

In [8]:
# 8.1 Market data feed ABCs

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Iterable, Callable


@dataclass
class Tick:
    symbol: str
    price: float
    timestamp: float  # seconds since epoch (simplified)


class MarketDataFeed(ABC):
    @abstractmethod
    def subscribe(self, symbol: str, callback: Callable[[Tick], None]) -> None:
        """Subscribe a callback to ticks for a symbol."""
        raise NotImplementedError

    @abstractmethod
    def start(self) -> None:
        """Start emitting ticks (synchronously in this simple example)."""
        raise NotImplementedError


@dataclass
class SimulatedFeed(MarketDataFeed):
    ticks: Iterable[Tick]

    def __post_init__(self) -> None:
        self._subscribers: dict[str, list[Callable[[Tick], None]]] = {}

    def subscribe(self, symbol: str, callback: Callable[[Tick], None]) -> None:
        self._subscribers.setdefault(symbol, []).append(callback)

    def start(self) -> None:
        for tick in self.ticks:
            for cb in self._subscribers.get(tick.symbol, []):
                cb(tick)


class LiveFeed(MarketDataFeed):
    """Placeholder for a production feed (e.g. WebSocket client).

    We'll just show the interface here.
    """

    def subscribe(self, symbol: str, callback: Callable[[Tick], None]) -> None:
        print(f"[LiveFeed] Subscribing to {symbol} (not actually implemented)")

    def start(self) -> None:
        print("[LiveFeed] Starting live feed (not actually implemented)")


In [9]:
# 8.2 Risk engine ABC

from abc import ABC, abstractmethod
from dataclasses import dataclass, field


class RiskEngine(ABC):
    @abstractmethod
    def check_order(self, order: AbstractOrder) -> None:
        """Raise if order violates risk limits."""
        raise NotImplementedError

    @abstractmethod
    def on_tick(self, tick: Tick) -> None:
        """Optional hook to update risk metrics from market data."""
        raise NotImplementedError


@dataclass
class MaxNotionalRiskEngine(RiskEngine):
    max_notional: float
    _last_prices: dict[str, float] = field(default_factory=dict)

    def check_order(self, order: AbstractOrder) -> None:
        symbol = order.instrument.symbol
        last_price = self._last_prices.get(symbol)
        if last_price is None:
            # In a real system, you'd fetch or require a price.
            print(f"[Risk] No price for {symbol}, allowing order but logging warning.")
            return

        # Notional exposure = last price * quantity
        notional = last_price * order.quantity
        if notional > self.max_notional:
            raise ValueError(
                f"Order notional {notional:.2f} exceeds max {self.max_notional:.2f}"
            )
        print(
            f"[Risk] Order for {symbol} with notional {notional:.2f} within limit {self.max_notional:.2f}"
        )

    def on_tick(self, tick: Tick) -> None:
        # Maintain latest price per symbol; used by check_order
        self._last_prices[tick.symbol] = tick.price


# 8.3 Wiring feed + risk engine + orders

# Simulated prices for AAPL
sim_ticks = [
    Tick("AAPL", price=180.0, timestamp=0.0),
    Tick("AAPL", price=182.0, timestamp=1.0),
]

feed = SimulatedFeed(sim_ticks)
risk_engine = MaxNotionalRiskEngine(max_notional=20_000)

# Subscribe risk engine to the feed so it sees ticks
feed.subscribe("AAPL", risk_engine.on_tick)

# Start feed (populate prices synchronously)
feed.start()

inst = EquityInstrument("AAPL")

small_order = MarketOrderCS(inst, side="BUY", quantity=50)   # 50 * 182 = 9100
big_order = MarketOrderCS(inst, side="BUY", quantity=200)    # 200 * 182 = 36400

for o in (small_order, big_order):
    print("Checking:", o.description)
    try:
        risk_engine.check_order(o)
    except ValueError as e:
        print("  RISK REJECTED:", e)
    else:
        print("  RISK OK")


Checking: BUY 50 AAPL @ market
[Risk] Order for AAPL with notional 9100.00 within limit 20000.00
  RISK OK
Checking: BUY 200 AAPL @ market
  RISK REJECTED: Order notional 36400.00 exceeds max 20000.00


### 8.4 How ABCs help here

- `MarketDataFeed` defines a generic interface for **any** price source.
- `RiskEngine` defines a generic interface for **any** risk policy.
- Concrete classes (`SimulatedFeed`, `LiveFeed`, `MaxNotionalRiskEngine`) implement the details.
- Your order-handling pipeline can depend only on:
  - `AbstractOrder`
  - `MarketDataFeed`
  - `RiskEngine`

This decouples **data acquisition**, **risk logic**, and **order representation** while
keeping the contracts explicit and enforceable.

## 9. Design Tips, Pitfalls, and Best Practices

### 9.1 When should you use an ABC?

- **Use an ABC when**:
  - You have multiple concrete classes that should share a **common interface**.
  - You want to **enforce** that certain methods/properties exist.
  - You want a place to put **shared logic** and **documentation** for that interface.
  - You are building a **plugin architecture** or **extensible framework**.

- **You might not need an ABC when**:
  - You have only one concrete implementation and no foreseeable variants.
  - Simple duck typing with **type hints and tests** is sufficient.
  - The interface is very small and informal.

### 9.2 ABCs vs Protocols (typing)

- **ABCs**:
  - Runtime concept (`isinstance`, `issubclass`).
  - Can include **concrete methods** and state.
  - Can **prevent instantiation** when abstract methods are missing.

- **Protocols** (from `typing` / `typing_extensions`):
  - Primarily for **static type checking**.
  - Are structural: any class with the right methods matches the protocol.
  - Do **not** enforce anything at runtime.

A common pattern:

- Use an **ABC** to define and document the core interface and shared logic.
- Optionally define a **Protocol** for more flexible, structural static typing.

### 9.3 Common pitfalls

- **Over-abstracting**: Don't create ABCs for every small thing. Start concrete, extract ABCs when you see clear variation patterns.
- **Leaky abstractions**: If every subclass needs to override most of the base behavior, your ABC might be too opinionated.
- **Tight coupling**: Keep ABCs focused on a single **responsibility**. Don't mix unrelated concerns into one base class.

### 9.4 Practical guidelines for trading-style systems

- Define ABCs for **core domains**:
  - Orders, positions, instruments, market data feeds, risk engines.
- Use mixins for **cross-cutting capabilities**:
  - Cancellable, risk-checkable, loggable, serializable.
- Use template methods to encode **lifecycles**:
  - Order submission, position lifecycle, risk evaluation cycles.
- Keep concrete classes as **thin as possible**, delegating shared logic to ABCs
  and leaving only genuinely variable behavior in subclasses.

### 9.5 How to practice

To deepen your understanding:

- **Exercise 1**: Add an `IcebergOrder` and `TrailingStopOrder` to the `AbstractOrder` hierarchy.
- **Exercise 2**: Extend `Position` to handle **realized PnL** and **multiple legs** (e.g., spreads).
- **Exercise 3**: Implement a new `RiskEngine` that enforces **max daily loss** and **per-symbol limits**.
- **Exercise 4**: Implement a `FileReplayFeed` that reads ticks from a CSV and implements `MarketDataFeed`.

Try to:

- Start from the **abstract interface**.
- Decide what must be **enforced** vs. what can be left to convention.
- Add concrete implementations and see how easily they plug into the rest of the system.
